In [1]:
import pandas as pd
import nltk
import string
from nltk.corpus import stopwords
nltk.download("stopwords")
from sentence_transformers import SentenceTransformer, util


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/manjunathpopuri/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
def preprocess(text):
    text = text.lower()
    text = text.replace(string.punctuation,"")
    text = nltk.word_tokenize(text)
    stop_words = set(stopwords.words("english"))
    text = [word for word in text if word not in stop_words]
    print(text)
    return " ".join(text)

In [3]:
def answer(model, user_question, data):

    embedding1 = model.encode(user_question)
    embedding2 = model.encode(list(data["question"]))

    top_5 = util.semantic_search(embedding1,embedding2, top_k = 5)
    return top_5[0]

In [4]:
if __name__ == "__main__":
    print("Enter Your Question:")
    user_input = input()
    print(user_input)
    text = preprocess(user_input)
    data = pd.read_csv("../Datasets/medquad.csv")
    model = SentenceTransformer("All-miniLM-L6-V2")

    ans = answer(model, text, data)

    for sentence in ans:
        print(f"Senetnce:{data["question"][sentence["corpus_id"]]} \n score = {sentence["score"]} \n Answer : {data["answer"][sentence["corpus_id"]]}\n Focus Area = {data["focus_area"][sentence["corpus_id"]]} \n Source = {data["source"][sentence["corpus_id"]]} \n")
    

Enter Your Question:
how do cough occours?
['cough', 'occours', '?']
Senetnce:What is (are) Cough ? 
 score = 0.5163505673408508 
 Answer : Coughing is a reflex that keeps your throat and airways clear. Although it can be annoying, coughing helps your body heal or protect itself. Coughs can be either acute or chronic. Acute coughs begin suddenly and usually last no more than 2 to 3 weeks. Acute coughs are the kind you most often get with a cold, flu, or acute bronchitis. Chronic coughs last longer than 2 to 3 weeks. Causes of chronic cough include       - Chronic bronchitis    - Asthma    - Allergies    - COPD (chronic obstructive pulmonary disease)    - GERD (gastroesophageal reflux disease)     - Smoking    - Throat disorders, such as croup in young children    - Some medicines        Water can help ease your cough - whether you drink it or add it to the air with a steamy shower or vaporizer. If you have a cold or the flu, antihistamines may work better than non-prescription cough me

In [2]:
import spacy

med_keywords_finder = spacy.load("en_core_sci_sm")

med_keywords_finder("Tell me more about heart attacks?").ents

(Tell, heart attacks)

In [5]:
import requests

# ===============================
# 1. Get OAuth2 Token
# ===============================
token_endpoint = 'https://icdaccessmanagement.who.int/connect/token'
client_id = 'e0cfbc3e-226a-41b4-92e2-90dca7ce3fc7_59347892-6a14-4f0b-a575-a456a1743a69'        # replace with your client_id
client_secret = 'GtSty9zkVKH9WP/oRKz1KBjQMHgUuVmbLexaMfdV3dE='  # replace with your client_secret
scope = 'icdapi_access'
grant_type = 'client_credentials'

payload = {
    'client_id': client_id,
    'client_secret': client_secret,
    'scope': scope,
    'grant_type': grant_type
}

# get the token
response = requests.post(token_endpoint, data=payload, verify=True)  # set verify=False only if necessary
response.raise_for_status()
token = response.json()['access_token']
print("Access Token obtained ✅")

# ===============================
# 2. Search ICD entities
# ===============================
def search_icd(term):
    uri = 'https://icd.who.int/icd/api/search'
    headers = {
        'Authorization': f'Bearer {token}',
        'Accept': 'application/json',
        'Accept-Language': 'en',
        'API-Version': 'v2'
    }
    params = {'q': term, 'language': 'en'}
    
    r = requests.get(uri, headers=headers, params=params)
    r.raise_for_status()
    data = r.json()
    
    results = []
    for item in data.get('destinationEntities', []):
        results.append({
            'title': item.get('title'),
            'code': item.get('code'),
            'definition': item.get('definition'),
            'synonyms': item.get('synonyms', [])
        })
    return results

# ===============================
# 3. Test with a query
# ===============================
query = "Heart"
icd_results = search_icd(query)
for r in icd_results:
    print(f"Title: {r['title']}")
    print(f"Code: {r['code']}")
    print(f"Definition: {r['definition']}")
    print(f"Synonyms: {r['synonyms']}")
    print("-"*50)

Access Token obtained ✅


HTTPError: 404 Client Error: Not Found for url: https://icd.who.int/icd/api/search?q=Heart&language=en

In [8]:
import requests

token_endpoint = 'https://icdaccessmanagement.who.int/connect/token'
client_id = 'e0cfbc3e-226a-41b4-92e2-90dca7ce3fc7_59347892-6a14-4f0b-a575-a456a1743a69'        # replace with your client_id
client_secret = 'GtSty9zkVKH9WP/oRKz1KBjQMHgUuVmbLexaMfdV3dE='  # replace with your client_secret
scope = 'icdapi_access'
grant_type = 'client_credentials'


# get the OAUTH2 token

# set data to post
payload = {'client_id': client_id, 
	   	   'client_secret': client_secret, 
           'scope': scope, 
           'grant_type': grant_type}
           
# make request
r = requests.post(token_endpoint, data=payload, verify=True).json()
token = r['access_token']


# access ICD API

uri = 'https://id.who.int/icd/entity'

# HTTP header fields to set
headers = {'Authorization':  'Bearer '+token, 
           'Accept': 'application/json', 
           'Accept-Language': 'en',
	   'API-Version': 'v2'}
           
# make request           
r = requests.get(uri, headers=headers, verify=True)

# print the result
print (r.text)			

{"@context":"http://id.who.int/icd/contexts/contextForTopLevel.json","@id":"http://id.who.int/icd/entity","title":{"@language":"en","@value":"WHO Family of International Classifications Foundation"},"releaseId":"2025-01","availableLanguages":["ar","cs","en","es","fr","kk","la","pt","ru","sk","sv","tr","uz","zh"],"releaseDate":"2025-01-24","child":["http://id.who.int/icd/entity/448895267","http://id.who.int/icd/entity/1405434703"],"browserUrl":"https://icd.who.int/browse/2025-01/foundation/en"}


In [ ]:
import spacy
import requests

# 1. Extract Medical Terms using scispaCy
def extract_medical_terms(text):
    nlp = spacy.load("en_core_sci_sm")
    doc = nlp(text)
    # Returns a list of extracted entities (e.g., ['heart attacks'])
    return [ent.text for ent in doc.ents]

# 2. Get OAUTH2 Token from WHO
def get_icd_access_token(client_id, client_secret):
    token_url = "https://icdaccessmanagement.who.int/connect/token"
    payload = {
        'client_id': client_id,
        'client_secret': client_secret,
        'scope': 'icdapi_access',
        'grant_type': 'client_credentials'
    }
    r = requests.post(token_url, data=payload)
    return r.json().get('access_token')

# 3. Search ICD-11 for the extracted terms
def get_icd_codes(term, token):
    search_url = f"https://id.who.int/icd/release/2025/en/search?q={term.replace(' ', '+')}"
    headers = {
        'Authorization': f'Bearer {token}',
        'Accept': 'application/json',
        'Accept-Language': 'en',
        'API-Version': 'v2'
    }
    response = requests.get(search_url, headers=headers)
    results = response.json().get('destinationEntities', [])
    return [(res.get('theCode'), res.get('title')) for res in results[:3]]

# Execution
QUESTION = "heart attacks?"
CLIENT_ID = 'e0cfbc3e-226a-41b4-92e2-90dca7ce3fc7_59347892-6a14-4f0b-a575-a456a1743a69'
CLIENT_SECRET = 'GtSty9zkVKH9WP/oRKz1KBjQMHgUuVmbLexaMfdV3dE='

# client_id = 'e0cfbc3e-226a-41b4-92e2-90dca7ce3fc7_59347892-6a14-4f0b-a575-a456a1743a69'        # replace with your client_id
# client_secret = 'GtSty9zkVKH9WP/oRKz1KBjQMHgUuVmbLexaMfdV3dE='  # replace with your client_secret

terms = extract_medical_terms(QUESTION)
token = get_icd_access_token(CLIENT_ID, CLIENT_SECRET)

for term in terms:
    print(f"Medical Term Found: {term}")
    codes = get_icd_codes(term, token)
    for code, title in codes:
        print(f" - ICD-11 Code: {code} | Title: {title}")


Medical Term Found: Talk


JSONDecodeError: Expecting value: line 1 column 1 (char 0)